Module importing

In [ ]:
import sys
import torch
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim
import scipy.signal as signal
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader,TensorDataset,random_split,SubsetRandomSampler, ConcatDataset

sys.path.append('D:/ppg_project/code/model_build/my_tool/model')
from MLP import MLP
sys.path.append('D:/ppg_project/code/model_build/my_tool/tool/data_load')
from temp_load import TemDataset

: 

Data prepairing

In [ ]:
from torch.utils.data import DataLoader

train_dataset = TempDataset("output_dataset/train")
val_dataset   = TempDataset("output_dataset/validate")
test_dataset  = TempDataset("output_dataset/test")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

Training

In [ ]:
latent_dim = 100
G = MLP(para={
            "dim":               [[latent_dim,256],[256,512],[512,1024],[1024,100]],
            "Activate function": "ReLU"
        })
    
D = MLP(para={
            "dim":               [[100,1024],[1024,512],[512,256],[256,1]],
            "Activate function": "LeakyReLU"
        })
device = "cuda" if torch.cuda.is_available() else "cpu"
if device:
    G.to(device)
    D.to(device)
    
learning_rate = 0.0002
num_epochs = 200
    
opt_G = torch.optim.Adam(G.parameters(), lr=learning_rate)
opt_D = torch.optim.Adam(D.parameters(), lr=learning_rate)

Tensor = torch.cuda.FloatTensor if device else torch.FloatTensor
num_epochs = 200

for epoch in range(num_epochs):
    # ============ TRAINING PHASE ============
    G.train()
    D.train()

    train_d_loss = 0
    train_g_loss = 0

    for i, (real_signals, _) in enumerate(train_loader):

        real_signals = real_signals.float().to(device)
        batch_size = real_signals.shape[0]

        # ---- Train Generator ----
        opt_G.zero_grad()

        z = torch.randn(batch_size, latent_dim, device=device)
        fake_signals = G(z)

        g_loss = bce(D(fake_signals),
                     torch.ones(batch_size, 1, device=device))

        g_loss.backward()
        opt_G.step()

        # ---- Train Discriminator ----
        opt_D.zero_grad()

        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        d_real = D(real_signals)
        d_fake = D(fake_signals.detach())

        loss_real = bce(d_real, real_labels)
        loss_fake = bce(d_fake, fake_labels)

        d_loss = loss_real + loss_fake
        d_loss.backward()
        opt_D.step()

        train_d_loss += d_loss.item()
        train_g_loss += g_loss.item()

    train_d_loss /= len(train_loader)
    train_g_loss /= len(train_loader)

    # ============ VALIDATION PHASE ============
    G.eval()
    D.eval()

    val_d_loss = 0
    val_g_loss = 0

    with torch.no_grad():
        for real_signals, _ in val_loader:

            real_signals = real_signals.float().to(device)
            batch_size = real_signals.shape[0]

            z = torch.randn(batch_size, latent_dim, device=device)
            fake_signals = G(z)

            # Discriminator validation loss
            d_real = D(real_signals)
            d_fake = D(fake_signals)

            loss_real = bce(d_real,
                            torch.ones(batch_size, 1, device=device))
            loss_fake = bce(d_fake,
                            torch.zeros(batch_size, 1, device=device))

            val_d_loss += (loss_real + loss_fake).item()

            # Generator validation loss
            val_g_loss += bce(D(fake_signals),
                              torch.ones(batch_size, 1, device=device)).item()

    val_d_loss /= len(val_loader)
    val_g_loss /= len(val_loader)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train  | D_loss: {train_d_loss:.4f}  G_loss: {train_g_loss:.4f}")
    print(f"Valid  | D_loss: {val_d_loss:.4f}  G_loss: {val_g_loss:.4f}\n")


Performance

In [ ]:
G.eval() # Set the generator to evaluation mode
num_samples_to_plot = 5 # Number of real and fake samples to plot

with torch.no_grad():
    # Get a batch of real signals from the test loader
    for real_signals, _ in test_loader:
        real_signals = real_signals.float().to(device)
        batch_size = real_signals.shape[0]

        # Generate fake signals from random noise
        z = torch.randn(batch_size, latent_dim, device=device)
        fake_signals = G(z)

        # Plot a few real and generated samples side-by-side
        for i in range(min(num_samples_to_plot, batch_size)):
            plt.figure(figsize=(10, 4))
            plt.plot(real_signals[i].cpu().numpy(), label='Real Signal')
            plt.plot(fake_signals[i].cpu().numpy(), label='Generated Signal')
            plt.title(f'Sample {i+1} - Real vs Generated Signals')
            plt.xlabel('Time Steps')
            plt.ylabel('Amplitude')
            plt.legend()
            plt.grid(True)
            plt.show()
        break # Only plot one batch for brevity